# Création du MPC

### Imports des packages + annotation type flexible

In [16]:
from __future__ import annotations
from typing import Any, Callable, Dict
import time

### Distinguer les erreur MPC des erreurs autres (api ect...)

In [17]:
class MCPError(Exception):
    pass

### Test

In [18]:
def validate_tool_call(payload: dict) -> None:
    if payload.get("type") != "tool_call":
        raise MCPError("Invalid type (expected tool_call)")
    if not payload.get("id"):
        raise MCPError("Missing id")
    if not payload.get("tool"):
        raise MCPError("Missing tool")
    if not isinstance(payload.get("args", {}), dict):
        raise MCPError("args must be an object")


### Routeur MCP

In [19]:
class MCPRouter:
    def __init__(self):
        self.tools: Dict[str, Callable[..., Any]] = {}

    def register(self, name: str, fn: Callable[..., Any]) -> None:
        self.tools[name] = fn

    def run(self, call: dict) -> dict:
        validate_tool_call(call)

        tool_name = call["tool"]
        call_id = call["id"]
        args = call.get("args", {})

        if tool_name not in self.tools:
            return {
                "type": "tool_result",
                "id": call_id,
                "tool": tool_name,
                "ok": False,
                "data": None,
                "error": f"Unknown tool: {tool_name}",
                "meta": {"elapsed_ms": 0, "cached": False},
            }

        start = time.time()
        try:
            data = self.tools[tool_name](**args)
            elapsed = int((time.time() - start) * 1000)
            return {
                "type": "tool_result",
                "id": call_id,
                "tool": tool_name,
                "ok": True,
                "data": data,
                "error": None,
                "meta": {"elapsed_ms": elapsed, "cached": False},
            }
        except Exception as e:
            elapsed = int((time.time() - start) * 1000)
            return {
                "type": "tool_result",
                "id": call_id,
                "tool": tool_name,
                "ok": False,
                "data": None,
                "error": str(e),
                "meta": {"elapsed_ms": elapsed, "cached": False},
            }